In [10]:
#%pip install plotnine


In [1]:
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from plotnine import ggplot, aes, geom_point, facet_grid, labs, scale_y_continuous, labeller, as_labeller, scale_x_continuous
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [2]:
# read in data
data = pd.read_csv("../data/rl_policies/tqc_clean.csv")

In [3]:
# get rid of t = 0:
data = data[data["t"] > 0]

In [4]:
# remove anomolous biomass data
data = data[(data['biomass'] != -1) & (data['biomass'] <= -0.46)]
data = data.iloc[5:]

In [5]:
# subset to relevant columns
subset = data[['months', 'act0', 'act1', 'CPUE', 'biomass']]

In [6]:
# loop through months and actions to cluster data
months = subset['months'].unique()
actions = ['act0', 'act1']
all_centroids = []
all_labeled = []

for month in months:
    data_month = subset[subset['months'] == month].drop(columns=['months'])
    
    for action in actions:
        other_action = 'act1' if action == 'act0' else 'act0'
        X = data_month.drop(columns=[other_action])
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        if month in [4, 5, 6, 7, 8] and action == 'act0':
            k = 1
        else:
            k = 3
        
        kmeans = KMeans(n_clusters=k, n_init=30, random_state=42).fit(X_scaled)
        
        # build labeled chunk
        chunk = X.copy().reset_index(drop=True)
        chunk['month'] = month
        chunk['action'] = action
        chunk['cluster'] = kmeans.labels_
        all_labeled.append(chunk)
        
        # store centroids
        centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
        centroids_df = pd.DataFrame(centroids_original, columns=X.columns)
        centroids_df['month'] = month
        centroids_df['action'] = action
        centroids_df['cluster'] = range(k)
        all_centroids.append(centroids_df)



In [7]:
labeled = pd.concat(all_labeled, ignore_index=True)
centroids = pd.concat(all_centroids, ignore_index=True)

In [7]:
labeled.head()

,act0,CPUE,biomass,month,action,cluster,act1
0,-0.964028,-0.981809,-0.879119,8,act0,0,NaN
1,-0.913234,-0.981059,-0.695920,8,act0,0,NaN
2,-0.918988,-0.975811,-0.739296,8,act0,0,NaN
3,-0.915355,-0.987716,-0.526255,8,act0,0,NaN
4,-0.916157,-0.989578,-0.536008,8,act0,0,NaN


In [8]:
centroids.head()

,act0,CPUE,biomass,month,action,cluster,act1,centroid_act
0,-0.924725,-0.987401,-0.655309,8,act0,0,NaN,-0.924725
1,NaN,-0.985756,-0.778625,8,act1,0,-0.289208,-0.289208
2,NaN,-0.990544,-0.603308,8,act1,1,0.326378,0.326378
3,NaN,-0.966580,-0.836234,8,act1,2,-0.153271,-0.153271
4,0.959271,-0.853993,-0.851572,9,act0,0,NaN,0.959271


In [19]:
# save labeled data and centroids
labeled.to_csv("../data/cluster/labeled.csv", index=False)
centroids.to_csv("../data/cluster/centroids.csv", index=False)


In [11]:
def find_closest_actions(CPUE, biomass, month, centroids):
    results = {}
    for action in ['act0', 'act1']:
        subset = centroids[(centroids['month'] == month) & (centroids['action'] == action)].copy()
        subset['dist'] = ((subset['CPUE'] - CPUE)**2 + (subset['biomass'] - biomass)**2)**0.5
        closest = subset.loc[subset['dist'].idxmin()]
        results[action] = closest[action]
    return results['act0'], results['act1']


class CentroidAgent:
    """Agent that selects actions by finding the nearest centroid for the current observation.

    Expects observations of the form {"crabs": np.array([CPUE, biomass]), "months": month},
    i.e. observation_type='count-biomass-time'.
    """
    def __init__(self, centroids: pd.DataFrame, env):
        self.centroids = centroids
        self.env = env

    def predict(self, observation, **kwargs):
        CPUE = float(observation["crabs"][0])
        biomass = float(observation["crabs"][1])
        month = int(observation["months"])
        act0, act1 = find_closest_actions(CPUE, biomass, month, self.centroids)
        return np.array([act0, act1], dtype=np.float32), {}

# example usage:
# centroid_agent = CentroidAgent(centroids=centroids, env=evalEnv)
# centroid_plot_agent = plot_agent(env_sim_df=None,
#                                  agent_name='centroid_agent',
#                                  env=evalEnv,
#                                  agent=centroid_agent,
#                                  save_dir='.')

In [9]:
# add a general action column to centroid data
centroids['centroid_act'] = centroids['act0'].combine_first(centroids['act1'])

In [10]:
centroids.head()

,act0,CPUE,biomass,month,action,cluster,act1,centroid_act
0,-0.924725,-0.987401,-0.655309,8,act0,0,NaN,-0.924725
1,NaN,-0.985756,-0.778625,8,act1,0,-0.289208,-0.289208
2,NaN,-0.990544,-0.603308,8,act1,1,0.326378,0.326378
3,NaN,-0.966580,-0.836234,8,act1,2,-0.153271,-0.153271
4,0.959271,-0.853993,-0.851572,9,act0,0,NaN,0.959271


In [14]:
# merge labeled to centroids
merged = labeled.merge(centroids, on=['month', 'action', 'cluster'], how='left')

In [15]:
month_names = {"4": "Apr", "5": "May",
               "6": "June", "7": "July", "8": "Aug",
               "9": "Sep", "10": "Oct"}
action_names = {"act0": "Minnow traps", "act1": "Fukui traps"}

month_order = ["Apr", "May", "June", "July", "Aug", "Sep", "Oct"]

plot_data = merged.copy()
plot_data['month'] = pd.Categorical(
    plot_data['month'].astype(str).map(month_names),
    categories=month_order,
    ordered=True
)
plot_data['action'] = plot_data['action'].map(action_names)

# add a general action column
#plot_data['act'] = plot_data['act0'].combine_first(plot_data['act1'])



In [20]:
# save plot data
plot_data.to_csv("../data/cluster/centroid_plot_data.csv", index=False)

In [17]:
p = (
    ggplot(plot_data, aes(x='biomass_x', y='CPUE_x', color='factor(cluster)'))
    + geom_point(alpha=0.4, size=0.5)
    + facet_grid('action ~ month')
    + labs(color='Cluster')
)
p.save('../figures/clustering.png', dpi=300, height = 3, width = 8)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: ../figures/clustering.png


In [18]:
p = (
    ggplot(plot_data, aes(x='biomass_x', y='CPUE_x', color='centroid_act'))
    + geom_point(alpha=0.4, size=0.5)
    + facet_grid('action ~ month')
    + labs(color='Cluster')
)
p.save('../figures/clustering_act.png', dpi=300, height = 3, width = 8)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: ../figures/clustering_act.png
